# 01 — Data Exploration

Bulk **tumor vs normal** gene expression across three independent GEO platforms
(lung GSE31210, colorectal GSE39582, breast GSE45827). This notebook inspects the
merged matrix, the ≈18:1 class imbalance, and the **batch == class** confounding that
motivated the leave-one-dataset-out (LODO) protocol.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import yaml
sys.path.insert(0, str(Path.cwd()))
from src.utils import load_config, set_seed

cfg = load_config()
set_seed(cfg["seed"])
print("seed:", cfg["seed"])


In [ ]:
# Load the merged tumor/normal matrix if present (data/ is gitignored, not committed).
p = Path("data/processed/tumor_normal.npy")
lp = Path("data/processed/tumor_normal_labels.npy")
if p.exists() and lp.exists():
    X = np.load(p)
    y = np.load(lp)
    print("shape:", X.shape)
    uniq, cnt = np.unique(y, return_counts=True)
    print("labels / counts:", dict(zip(uniq.tolist(), cnt.tolist())))
    print("imbalance (tumor:normal) ~", round(cnt.max() / max(cnt.min(), 1), 1), ": 1")
else:
    print("data/processed/ not present (gitignored). Regenerate with:")
    print("  python -m src.tumor_normal")


## Batch vs class confounding

In the original multi-class task, each GSE series is a single cancer type, so the
platform and the label are 1:1 aliased. Random splits then reach macro-F1 = 1.0 by
memorizing the platform, while LODO collapses to macro-F1 = 0.0. See the PCA projection
of batch identity before any correction:

![PCA by batch (before correction)](../results/figures/pca_by_batch_before.png)
